In [ ]:
import os
import torch
import numpy as np
from PIL import Image
import tifffile as tiff
from utils.validation import denoise_folder, calc_metrics
from utils.filters import (
    bm3d_denoise, 
    tv_denoise, 
    gaussian_denoise, 
    median_denoise, 
    wavelet_denoise, 
    nlm_denoise
)
from denoisers import build_nn_models, care_denoise, build_care, nn_denoise, rauden_denoise
from models.rauden import RAUDen
from models.baselines.dncnn import DnCNN
from models.baselines.unet import UnetN2N
from pathlib import Path
from utils.validation import extract_segments_from_json, calculate_mean_intensity_in_folders

In [ ]:
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
X_SCALE = 0.022
Z_SCALE = 0.1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# sticks
STICKS_FOLDER = "validation/synthetic/sticks"
GT_STICKS = os.path.join(STICKS_FOLDER, "clean_sticks")
NOISY_STICKS = os.path.join(STICKS_FOLDER, "noisy_sticks")
FILTERS_STICKS = os.path.join(STICKS_FOLDER, "classic_denoised_sticks")
FILTERS_METRICS = os.path.join(STICKS_FOLDER, "filter_sticks_metrics")
DL_STICKS = os.path.join(STICKS_FOLDER, "dl_denoised_sticks")
DL_METRICS = os.path.join(STICKS_FOLDER, "dl_sticks_metrics")
GT_SEGMENTS = "validation/synthetic/sticks/gt_segments"

# dendrites
NOISY_DENDRITES = "validation/real/dendrites/noisy"
NOISY_DENDRITES_METRICS = "validation/real/dendrites/noisy"
FILTERS_DENDRITES = "validation/real/dendrites/filter_denoised"
FILTERS_DENDRITES_METRICS = "validation/real/dendrites/filter_dendrites_metrics"
DL_DENDRITES = "validation/real/dendrites/dl_denoised"
DL_DENDRITES_METRICS = "validation/real/dendrites/dl_dendrites_metrics"

# pretrained weights
RAUDEN_MODEL_PATH = "./experiments/rauden/Sep_02_21_28/checkpoints/model_epoch200.pth"
DNCNN_MODEL_PATH = 'experiments/pretrained/dncnn/checkpoints/model_epoch400.pth'
N2N_MODEL_PATH = 'experiments/pretrained/n2n/checkpoints/model_epoch400.pth'
CARE_PATH = "care_weights"

In [ ]:
classic_methods = [
    ("GaussianFilter", lambda img: gaussian_denoise(img, sigma=3)),
    ("MedianFilter", lambda img: median_denoise(img, size=3)),
    ("Non-localMeansFilter", lambda img: nlm_denoise(img)),
    ("TotalVariationFilter", lambda img: tv_denoise(img, weight=0.08, n_iter_max=200,
                        voxel_size=(0.1, 0.022, 0.022), resample_order=1)),
    ("WaveletFilter", lambda img: wavelet_denoise(img,  wavelet='bior4.4', level=1)),
    ("BM3DFilter", lambda img: bm3d_denoise(img, sigma_psd=0.05, n_jobs=8))
]

In [ ]:
denoise_folder(
    input_folder=NOISY_STICKS,
    output_folder=FILTERS_STICKS,
    methods=classic_methods
)

In [ ]:
calc_metrics(
    gt_folder=GT_STICKS,
    denoised_root=FILTERS_STICKS,
    csv_dir=FILTERS_METRICS
)

In [ ]:
dncnn, n2n = build_nn_models(
    [DNCNN_MODEL_PATH, N2N_MODEL_PATH],
    device,
    [DnCNN(), UnetN2N(in_channels=1, out_channels=1)]
)
care = build_care(CARE_PATH)

In [ ]:
dl_methods = [
    ("CARE", lambda img: care_denoise(img, model=care)),
    ("DnCNN", lambda img: nn_denoise(img, model=dncnn)),
    ("Noise2Noise", lambda img: nn_denoise(img, model=n2n)),
    ("RCAN", lambda img: nn_denoise(img, model=rauden)),
    ("SRDTrans", lambda img: wavelet_denoise(img,  wavelet='bior4.4', level=1)),
    ("RAUDen", lambda img: rauden_denoise(img, model=rauden))
]

In [ ]:
denoise_folder(
    input_folder=NOISY_STICKS,
    output_folder=DL_STICKS,
    methods=dl_methods
)

In [ ]:
calc_metrics(
    gt_folder=GT_STICKS,
    denoised_root=DL_STICKS,
    csv_dir=DL_METRICS
)

In [ ]:
ref_root = Path(GT_SEGMENTS)
target_folder = Path("validation/synthetic/sticks/dl_denoised_sticks/RAUDen/poisson_1_gauss_0.2") # Noise2Noise
out_root = Path("validation/synthetic/sticks/dl_segments/RAUDen/poisson_1_gauss_0.2")

logs = extract_segments_from_json(ref_root, target_folder, out_root)
for line in logs:
    print(line)

### Synthetic live-calcium processing

In [ ]:
denoise_folder(
    input_folder="validation/synthetic/calcium/1/noisy",
    output_folder="validation/synthetic/calcium/1/dl_denoised",
    methods=dl_methods
)

### Confocal dendrites processing

In [ ]:
calc_metrics(
    denoised_root=NOISY_DENDRITES,
    csv_dir=NOISY_DENDRITES_METRICS,
    only_noisy=True
)

In [ ]:
denoise_folder(
    input_folder=NOISY_DENDRITES,
    output_folder=FILTERS_DENDRITES,
    methods=classic_methods
)

In [ ]:
calc_metrics(
    denoised_root=FILTERS_DENDRITES,
    csv_dir=FILTERS_DENDRITES_METRICS
)

In [ ]:
denoise_folder(
    input_folder=NOISY_DENDRITES,
    output_folder=DL_DENDRITES,
    methods=dl_methods
)

In [ ]:
calc_metrics(
    denoised_root=DL_DENDRITES,
    csv_dir=DL_DENDRITES_METRICS
)